## Cài đặt và Import các thư viện cần thiết

In [1]:
# Cài đặt thư viện (nếu chưa có)
!pip install -q ultralytics matplotlib opencv-python ipywidgets

import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
import numpy as np
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

## Khởi tạo Mô hình YOLO11

In [2]:
# Đường dẫn tới file trọng số tốt nhất (từ quá trình train trước đó)
MODEL_PATH = "../weights/best.pt" 

# Kiểm tra và Load model
if os.path.exists(MODEL_PATH):
    model = YOLO(MODEL_PATH)
    print("Đã load model YOLO11 thành công!")
else:
    print(f"Lỗi: Không tìm thấy file '{MODEL_PATH}'. Vui lòng copy file best.pt vào cùng thư mục với Notebook này.")

Đã load model YOLO11 thành công!


## Xây dựng hàm Pipeline

In [3]:
def object_detection_pipeline(image_input, conf_threshold=0.25):
    """
    Hàm xử lý Pipeline Object Detection nhận dạng rác thải.
    
    Input:
        - image_input: Bức ảnh đầu vào (Numpy array)
        - conf_threshold: Ngưỡng tự tin (confidence score) để lọc các nhiễu.
        
    Output:
        - result_img: Bức ảnh dưới dạng numpy array (Hệ màu RGB) đã được vẽ bounding boxes và nhãn.
        - raw_results: Đối tượng result gốc chứa tọa độ, tên nhãn (Dành cho Backend).
    """
    
    # 1. Đưa ảnh vào mô hình dự đoán (Sử dụng imgsz=640 giống với quá trình Train)
    results = model.predict(
        source=image_input, 
        conf=conf_threshold, 
        imgsz=640, 
        save=False,
        verbose=False # Tắt log cho giao diện sạch sẽ
    )
    
    result = results[0]
    
    # 2. Lấy ảnh đầu ra đã được model tự động vẽ bounding box, class, confidence
    img_bgr = result.plot()
    
    # 3. Chuyển đổi hệ màu sang RGB để hiển thị đúng màu trên Matplotlib
    result_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    return result_img, result

## Upload ảnh và Test

In [18]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import cv2

# 1. Tạo nút bấm Upload
upload_button = widgets.FileUpload(
    accept='image/*',  
    multiple=False,    
    description='Tải ảnh lên',
    button_style='info'
)

# 2. Dòng chữ hướng dẫn khi CHƯA CÓ ẢNH
guide_label = widgets.HTML(
    "<p style='color: #666; font-size: 13px; margin-top: 8px; font-style: italic;'>"
    "Vui lòng nhấn nút <b>'Tải ảnh lên'</b> ở trên để nhận diện rác thải."
    "</p>"
)

# 3. Các thành phần kết quả (Mặc định được ẨN ĐI lúc chưa có ảnh)
title_label = widgets.HTML(
    "<b style='font-size: 15px; text-align: center; display: block; margin-bottom: 8px;'>OUTPUT: Detected Image</b>"
)
title_label.layout.display = 'none' # Ẩn tiêu đề

image_display = widgets.Image(
    format='jpg',
    layout=widgets.Layout(width='100%', max_width='400px', display='none') # Ẩn ảnh (tránh icon vỡ)
)

filename_label = widgets.HTML()

# Cột trái (Ảnh & Tên file)
image_box = widgets.VBox([
    title_label,
    image_display,
    filename_label
], layout=widgets.Layout(width='50%', align_items='center'))

# Cột phải (Text bóc tách)
text_output = widgets.Output(layout=widgets.Layout(width='50%'))  
ui_layout = widgets.HBox([image_box, text_output])             

# 4. Xử lý sự kiện khi người dùng chọn ảnh
def on_file_upload(change):
    value = upload_button.value
    
    if value:
        # KHI ĐÃ CÓ ẢNH: Ẩn dòng hướng dẫn, Bật hiển thị kết quả lên
        guide_label.layout.display = 'none'
        title_label.layout.display = 'block'
        image_display.layout.display = 'block'
        
        # Xóa kết quả text cũ
        text_output.clear_output(wait=True)
        
        # Trích xuất dữ liệu ảnh
        if isinstance(value, tuple) or isinstance(value, list): 
            content = value[0]['content']
            filename = value[0].get('name', 'Uploaded Image')
        else: 
            filename = list(value.keys())[0]
            content = value[filename]['content']
            
        nparr = np.frombuffer(content, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        if img is None:
            with text_output:
                print("Error: Cannot read image file. Please upload another image!")
            return

        # Gọi hàm AI
        output_image, raw_data = object_detection_pipeline(img, conf_threshold=0.3)
        
        # Đẩy ảnh vào widget
        img_bgr = cv2.cvtColor(output_image, cv2.COLOR_RGB2BGR)
        _, encoded_img = cv2.imencode('.jpg', img_bgr)
        image_display.value = encoded_img.tobytes()
        
        # Cập nhật tên file
        filename_label.value = f"<span style='color: #555; font-size: 13px; font-weight: 500; text-align: center; display: block; margin-top: 6px;'>📁 File: <i>{filename}</i></span>"
        
        # Hiển thị Text bóc tách bên phải
        with text_output:
            print("="*45)
            print(" DETECTION RESULTS")
            print("="*45)
            
            if len(raw_data.boxes) == 0:
                print("No objects detected (conf >= 0.3)")
            else:
                for box in raw_data.boxes:
                    class_id = int(box.cls[0])
                    class_name = model.names[class_id]
                    conf = float(box.conf[0])
                    xyxy = box.xyxy[0].cpu().numpy().astype(int)
                    
                    print(f"🔹 Class: {class_name}")
                    print(f"   Confidence: {conf:.2%} | BBox: {xyxy}")
                    print("-" * 45)

# 5. Gắn sự kiện và Hiển thị
upload_button.observe(on_file_upload, names='value')
display(upload_button, guide_label, ui_layout)

FileUpload(value={}, accept='image/*', button_style='info', description='Tải ảnh lên')

HTML(value="<p style='color: #666; font-size: 13px; margin-top: 8px; font-style: italic;'>Vui lòng nhấn nút <b…